# Importing the required libraries

### Importing and Pandas configurations
Here we are importing the os and glob to read multiple datafiles to python code/env
Also setting up the pandas df display settings

In [47]:
import os
import glob
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

### Gathering the data
Gathering all the csv files from the given folder path and storing them in all files


In [48]:
#Grab all CSV paths from your target folder
folder_path = r"X:\Powerbi\Projects\Brazillian_ecommerce_analysis\Data"
all_files = glob.glob(os.path.join(folder_path, "*.csv"))


# Extracting the Structural data and Health profiles

### Defining the required dicts and lists and extracting data from files
Here the every csv dat is extracted using the for loop and storing the data files as pd dataframes in dict and lists the diffrent dictionaries are created for the different profiling you will see those in below cells


In [49]:
# 2. Extract structural data health profiles
file_profiles =[]
dfs =[]
column_def=[]
describe_prf=[]
null_count=[]
samples_f=[]
for file in all_files:
    try:
        # Load sample or full file depending on size
        df_temp = pd.read_csv(file) # Quick check on first 1000 rows with nrows=1000
        
        profile = {
            "File_Name": os.path.basename(file),
            "Row_Count": len(df_temp),
            "Column_Count": len(df_temp.columns),
            #"Columns": list(df_temp.columns),
            "Missing_Values_Total": df_temp.isnull().sum().sum()
        }
        column_prf={
            "File_Name": os.path.basename(file),
            "Column names": list(df_temp.columns)
        }
        nullcntf={
            "File_Name": os.path.basename(file),
            "Null columns": df_temp.isnull().sum()
        }
        des_prf={
            "File_Name": os.path.basename(file),
            "Describe": df_temp.describe(include="all")
        }
        samples={
            "File_Name": os.path.basename(file),
            "Samples": df_temp.sample(10)
        }

        
        file_profiles.append(profile)
        column_def.append(column_prf)
        null_count.append(nullcntf)
        describe_prf.append(des_prf)
        samples_f.append(samples)

        # Keep track for final master merge
        #dfs.append(pd.read_csv(file))
    except Exception as e:
        print(f"⚠️ Structural error found in file {os.path.basename(file)}: {e}")

### Profiling the shape and missing values
Here you can see the total No. of rows and columns along with the missing values of those files

In [50]:
# Validate if all files match column layouts
profile_df = pd.DataFrame(file_profiles)
profile_df

# 3. If clean, consolidate into a master staging dataframe
#if len(dfs) > 0:
#    master_df = pd.concat(dfs, ignore_index=True)
#    # Save a staging dataset
#    master_df.to_csv(os.path.join(folder_path, "master_staging.csv"), index=False)

,File_Name,Row_Count,Column_Count,Missing_Values_Total
0,olist_customers_dataset.csv,99441,5,0
1,olist_geolocation_dataset.csv,1000163,5,0
2,olist_orders_dataset.csv,99441,8,4908
3,olist_order_items_dataset.csv,112650,7,0
4,olist_order_payments_dataset.csv,103886,5,0
5,olist_order_reviews_dataset.csv,99224,7,145903
6,olist_products_dataset.csv,32951,9,2448
7,olist_sellers_dataset.csv,3095,4,0
8,product_category_name_translation.csv,71,2,0


### Verifyin columns
Here you can check if all the csv files are haivng the required columns as expected

In [51]:
column_df=pd.DataFrame(column_def)
column_df

,File_Name,Column names
0,olist_customers_dataset.csv,"[customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state]"
1,olist_geolocation_dataset.csv,"[geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state]"
2,olist_orders_dataset.csv,"[order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]"
3,olist_order_items_dataset.csv,"[order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]"
4,olist_order_payments_dataset.csv,"[order_id, payment_sequential, payment_type, payment_installments, payment_value]"
5,olist_order_reviews_dataset.csv,"[review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]"
6,olist_products_dataset.csv,"[product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]"
7,olist_sellers_dataset.csv,"[seller_id, seller_zip_code_prefix, seller_city, seller_state]"
8,product_category_name_translation.csv,"[product_category_name, product_category_name_english]"


### Null values ?
Here we can find where the actual null values exists in those files

In [52]:
for item in null_count:
    s = item["Null columns"]
    s = s[s > 0]

    if s.empty:
        pass
        #print(item["File_Name"])
        #print("No null values\n")
    else:
        print(item["File_Name"])
        display(s.to_frame("null_count"))
        #print(s)

olist_orders_dataset.csv


,null_count
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965


olist_order_reviews_dataset.csv


,null_count
review_comment_title,87656
review_comment_message,58247


olist_products_dataset.csv


,null_count
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


### Describing data
Here we describe the data to see if there are any out right anomalies that are need to be questioned

In [53]:
for item in describe_prf:
  print(item["File_Name"])
  display(item["Describe"])

olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
count,99441,99441,99441.000000,99441,99441
unique,99441,96096,NaN,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,NaN,sao paulo,SP
freq,1,17,NaN,15540,41746
mean,NaN,NaN,35137.474583,NaN,NaN
std,NaN,NaN,29797.938996,NaN,NaN
min,NaN,NaN,1003.000000,NaN,NaN
25%,NaN,NaN,11347.000000,NaN,NaN
50%,NaN,NaN,24416.000000,NaN,NaN
75%,NaN,NaN,58900.000000,NaN,NaN


olist_geolocation_dataset.csv


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
count,1.000163e+06,1.000163e+06,1.000163e+06,1000163,1000163
unique,NaN,NaN,NaN,8011,27
top,NaN,NaN,NaN,sao paulo,SP
freq,NaN,NaN,NaN,135800,404268
mean,3.657417e+04,-2.117615e+01,-4.639054e+01,NaN,NaN
std,3.054934e+04,5.715866e+00,4.269748e+00,NaN,NaN
min,1.001000e+03,-3.660537e+01,-1.014668e+02,NaN,NaN
25%,1.107500e+04,-2.360355e+01,-4.857317e+01,NaN,NaN
50%,2.653000e+04,-2.291938e+01,-4.663788e+01,NaN,NaN
75%,6.350400e+04,-1.997962e+01,-4.376771e+01,NaN,NaN


olist_orders_dataset.csv


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32951,32341,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
unique,32951,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,NaN,NaN,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,NaN,NaN,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,NaN,NaN,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,NaN,NaN,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,NaN,NaN,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000


olist_sellers_dataset.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state
count,3095,3095.000000,3095,3095
unique,3095,NaN,611,23
top,3442f8959a84dea7ee197c632cb2df15,NaN,sao paulo,SP
freq,1,NaN,694,1849
mean,NaN,32291.059451,NaN,NaN
std,NaN,32713.453830,NaN,NaN
min,NaN,1001.000000,NaN,NaN
25%,NaN,7093.500000,NaN,NaN
50%,NaN,14940.000000,NaN,NaN
75%,NaN,64552.500000,NaN,NaN


product_category_name_translation.csv


,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


### Sample data
Here is the sample data to understand the overview of existing data these values can change evey time we run the data difing cell above

In [54]:
for item in samples_f:
  print(item["File_Name"])
  display(item["Samples"])

olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
8654,c63812833200850d116bb2ba25ad634d,0e7605b7d6890bb5d98446a80b91bca8,2617,sao paulo,SP
14875,053c7943cf3218c20664c01973f729de,040405db5973c3f48155ed4280b29b86,6612,jandira,SP
43854,e895647c3d61dae1606151534adb3acc,1d4b7f1c226ef793c9b1c3db7fc9713e,64202,parnaiba,PI
83525,c3192b5f85bb0b0a52c4777dda01af09,81492b37a762bbce212320608f4fe497,11410,guaruja,SP
87122,a5acc83d0613fa082aee11996117121c,019d47513bb1164cc83fc2d6d7123ef8,48120,pojuca,BA
68780,a1d3be9164de0c24121b9188a1e4e1ca,2f415501f319de1f14dcbb8f67d402db,11420,guaruja,SP
90735,ff8f73eac7282a36535b5deef7360ae2,18170ac5a96bb8e48812cf402598b1df,53370,olinda,PE
87192,60cdb2401584f6851ca9e6254009a2b0,63e5029bb863ff666d1d40792f1cdcac,96085,pelotas,RS
71419,efc682b906113fd3d8d7bec178aca886,4b06970bf3132cb4456b75269c592e1a,96810,santa cruz do sul,RS
71677,d47e69f46624eccc55b349af80fc4c53,3377cc8994bab907b65292c92f06dcba,53625,igarassu,PE


olist_geolocation_dataset.csv


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
222504,9170,-23.711062,-46.511064,santo andre,SP
63816,3625,-23.514967,-46.512371,sao paulo,SP
68912,3978,-23.623591,-46.505577,sao paulo,SP
759677,65393,-4.313711,-46.460505,buriticupu,MA
866981,84060,-25.117311,-50.201013,ponta grossa,PR
614507,36328,-21.117565,-44.222625,santa cruz de minas,MG
259145,11660,-23.627950,-45.418636,caraguatatuba,SP
275968,12316,-23.320376,-45.941996,jacarei,SP
397246,19063,-22.132354,-51.444703,presidente prudente,SP
414742,20511,-22.933156,-43.244458,rio de janeiro,RJ


olist_orders_dataset.csv


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
7161,4b2042bfdcfa4cf33625cad29a15e107,fashion_bolsas_e_acessorios,38.0,236.0,2.0,200.0,24.0,9.0,16.0
32611,515363bc1bda36afb33bb9cdde0789f3,livros_interesse_geral,36.0,205.0,1.0,250.0,27.0,9.0,18.0
7390,29bcb163d2d9fdda52f39e9b84f2eec7,informatica_acessorios,47.0,517.0,1.0,900.0,35.0,15.0,15.0
31119,3cc8edea2678a63c9e01c2dd190a395d,eletrodomesticos,56.0,276.0,1.0,4000.0,27.0,13.0,17.0
22128,fadde2aec26938dd22add77b290d279d,utilidades_domesticas,57.0,614.0,1.0,4400.0,24.0,28.0,39.0
6641,fe7ad3e3828f32193dfbe6359f733a8f,brinquedos,48.0,609.0,3.0,823.0,25.0,28.0,28.0
4548,f77f18b15df7b13a76cac068e285eb78,instrumentos_musicais,48.0,283.0,1.0,350.0,27.0,6.0,20.0
13558,9b887ea9c5de0f08367e5bd0e48a4c0f,pet_shop,54.0,1043.0,1.0,680.0,32.0,29.0,22.0
3849,b833b995caddd97b8786370988e32ee4,utilidades_domesticas,54.0,351.0,2.0,200.0,20.0,15.0,15.0
3095,1468ae815f666819cd3fa15be0cc035a,eletronicos,58.0,870.0,1.0,575.0,26.0,26.0,26.0


olist_sellers_dataset.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state
1007,4049512728d969bec69f84088b012416,30310,belo horizonte,MG
364,c716e0b86ed568878475b60fbb6323ad,22783,rio de janeiro,SP
976,1fbe10c70e30765ed1de86d9f7e98409,9320,maua,SP
2940,3a3e46d10f74d8a6ca76e2b538f106a0,2522,sao paulo,SP
2215,0ea22c1cfbdc755f86b9b54b39c16043,35700,sete lagoas,MG
1335,c113b8df1fc375bc3d73d29c1b1144d1,8021,sao paulo,SP
2512,76623fde66e6b63b59440fd34160512d,7124,guarulhos,SP
1517,9a208dee8f95cfdf00760c4d627828ec,26125,belford roxo,RJ
941,5bf397d04035b5f45cc41e82ec106803,4144,sao paulo,SP
868,55c26bcb609f480eb7868594245febb5,14910,tabatinga,SP


product_category_name_translation.csv


,product_category_name,product_category_name_english
44,industria_comercio_e_negocios,industry_commerce_and_business
2,automotivo,auto
54,artigos_de_natal,christmas_supplies
5,esporte_lazer,sports_leisure
68,fraldas_higiene,diapers_and_hygiene
10,alimentos_bebidas,food_drink
23,malas_acessorios,luggage_accessories
61,musica,music
66,flores,flowers
47,moveis_quarto,furniture_bedroom


# Cleaning the data
By gathering all the data profile and info. The only process before we work on the data remained is cleaning it.

### Goals for now
* Check duplicates before dropping anything.
* why is customers id looks encryped in customers table and also why there are two keys one uniqe and another normal
* Found there are some zip_code_prefix with 4 digit values which is not possible replace it with original or check if there is 0 infront of those codes
* Fix types. Dates become real datetimes, and zip codes become zero-padded text.
* Orders. It checks the null delivery dates against order_status, and flags delivered orders with no delivery date. It also adds delivery_days, delivery_delay_days and is_late.
* why are the customers id and orders id matching did all the customers ordered only single time? *** wtf ***
* What is shipping limit date and order item id denotes why the ids looks encrypted
* what is freight value?
* How do the payment table work?
* review creation vs review answered date
* Nan Review titles should be converted to empty or ""?
* why there are duplicate review ids?
* Reviews. It removes repeated review_ids and adds a has_comment flag.
* Products. It fixes the lenght typos, fills the missing category with "unknown", and merges in the English names.
*Geolocation. It drops coordinates outside Brazil and collapses to one row per zip.
* Text and sanity checks. City names are lowercased and stripped of accents. It also checks payments and prices.
* Orphan-key checks between related tables.
* Final null check, then export to a Cleaned folder next to your Data folder.

### Analyzing the Files and noting down the key take aways
First step is to **keep the actual DataFrames** in a dict.
Do not `pd.concat` these tables into one master file: they have different columns and meanings. Keep them as separate tables (star schema) for Power BI.

In [55]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("pandas", pd.__version__)

DATA  = Path(r"X:\Powerbi\Projects\Brazillian_ecommerce_analysis\Data")
CLEAN = DATA.parent / "Cleaned"
CLEAN.mkdir(exist_ok=True)

short = {
    "olist_customers_dataset": "customers",
    "olist_geolocation_dataset": "geolocation",
    "olist_orders_dataset": "orders",
    "olist_order_items_dataset": "order_items",
    "olist_order_payments_dataset": "payments",
    "olist_order_reviews_dataset": "reviews",
    "olist_products_dataset": "products",
    "olist_sellers_dataset": "sellers",
    "product_category_name_translation": "category_translation",
}

zip_cols = {c: str for c in ["customer_zip_code_prefix",
                             "seller_zip_code_prefix",
                             "geolocation_zip_code_prefix"]}

d = {short[p.stem]: pd.read_csv(p, dtype=zip_cols)
     for p in DATA.glob("*.csv") if p.stem in short}

display({k: v.shape for k, v in d.items()})

pandas 3.0.6


{'customers': (99441, 5),
 'geolocation': (1000163, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'payments': (103886, 5),
 'reviews': (99224, 7),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'category_translation': (71, 2)}

## Duplicate check
Look before dropping anything. For each table this prints fully identical rows and, where a primary key exists, repeated key values.

In [56]:
keys = {"customers": "customer_id", "orders": "order_id", "reviews": "review_id",
        "products": "product_id", "sellers": "seller_id"}

for name, df in d.items():
    line = f"{name:22s} identical rows: {df.duplicated().sum():>7,}"
    if name in keys:
        line += f" | repeated {keys[name]}: {df[keys[name]].duplicated().sum():,}"
    print(line)

customers              identical rows:       0 | repeated customer_id: 0
geolocation            identical rows: 261,831
orders                 identical rows:       0 | repeated order_id: 0
order_items            identical rows:       0
payments               identical rows:       0
reviews                identical rows:       0 | repeated review_id: 814
products               identical rows:       0 | repeated product_id: 0
sellers                identical rows:       0 | repeated seller_id: 0
category_translation   identical rows:       0


## Understand the review duplicates before deciding
`review_id` repeats in this dataset. These three numbers tell you whether the same review is shared across several orders or whether rows are exact copies.
- If `(review_id, order_id)` repeats are near zero, one review covers several orders.
- If `order_id` repeats are above zero, some orders have more than one review.

In [57]:
r = d["reviews"]
print("repeated review_id            :", r["review_id"].duplicated().sum())
print("repeated (review_id, order_id):", r.duplicated(["review_id", "order_id"]).sum())
print("orders with more than 1 review:", r["order_id"].duplicated().sum())

repeated review_id            : 814
repeated (review_id, order_id): 0
orders with more than 1 review: 551


In [58]:
def zip5(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\.0$", "", regex=True)
             .str.zfill(5))

## Zip codes: diagnose, then fix
Brazilian zip prefixes have 5 digits. A 4-digit value is a code whose leading zero was lost (such as `01046` saved as `1046`).
First confirm the theory: short codes should belong almost entirely to São Paulo (`SP`), whose codes start with 0.

In [59]:
c = d["customers"]
short_zip = c[c["customer_zip_code_prefix"].astype("string").str.strip().str.len() == 4]
print("4-digit zips:", len(short_zip))
print(short_zip["customer_state"].value_counts().head())

4-digit zips: 0
Series([], Name: count, dtype: int64)


In [60]:
for name, col in [("customers", "customer_zip_code_prefix"),
                  ("sellers", "seller_zip_code_prefix"),
                  ("geolocation", "geolocation_zip_code_prefix")]:
    d[name][col] = zip5(d[name][col])

raw_geo_zips = set(d["geolocation"]["geolocation_zip_code_prefix"].dropna())

print({n: d[n][c].str.len().value_counts().to_dict()
       for n, c in [("customers", "customer_zip_code_prefix"),
                    ("sellers", "seller_zip_code_prefix"),
                    ("geolocation", "geolocation_zip_code_prefix")]})

{'customers': {np.int64(5): 99441}, 'sellers': {np.int64(5): 3095}, 'geolocation': {np.int64(5): 1000163}}


## Convert text columns to real datetimes
`read_csv` loads dates as text. `errors="coerce"` turns anything unparseable into `NaT` (missing date) instead of crashing.

In [61]:
date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
for name, cols in date_cols.items():
    for col in cols:
        d[name][col] = pd.to_datetime(d[name][col], errors="coerce")

d["orders"].dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Orders: do the null dates make sense?
A missing delivery date is expected for canceled or in-transit orders. A **delivered** order without a delivery date is a genuine data problem.

In [62]:
orders = d["orders"]

status_check = (orders.assign(no_delivery_date=orders["order_delivered_customer_date"].isna())
                      .groupby("order_status")["no_delivery_date"].agg(no_delivery_date="sum", orders="size"))
display(status_check)

bad = orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()]
print("delivered but missing delivery date:", len(bad))

,no_delivery_date,orders
order_status,,
approved,2,2
canceled,619,625
created,5,5
delivered,8,96478
invoiced,314,314
processing,301,301
shipped,1107,1107
unavailable,609,609


delivered but missing delivery date: 8


## Orders: derived columns for the dashboard
- `order_purchase_date`: date without time, used to link a Power BI date table.
- `delivery_days`: purchase to delivery.
- `delivery_delay_days`: positive means delivered after the promised date.
- `is_late`: 1 / 0, and left blank when the order was never delivered (no fake 0).
- Orders: derived columns and anomaly flags  (CHANGED: two new flags)
- `delivered_missing_date` = 1 for delivered orders with no delivery date (your 8).
- `canceled_with_delivery` = 1 for canceled orders that do have a delivery date (your 6).

Both stay in the data, so you can filter or exclude them in Power BI.
`orders = d["orders"]` is a reference, so the last line `d["orders"] = orders` is what stores the result. Keep it.

In [63]:
orders["order_purchase_date"] = orders["order_purchase_timestamp"].dt.normalize()

orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days.astype("Int16")
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"]
                                 - orders["order_estimated_delivery_date"]).dt.days.astype("Int16")

delay = orders["delivery_delay_days"]
orders["is_late"] = (delay > 0).astype("Int8").where(delay.notna())

# NEW: anomaly flags (flag, don't remove)
orders["delivered_missing_date"] = ((orders["order_status"] == "delivered")
                                    & orders["order_delivered_customer_date"].isna()).astype(int)
orders["canceled_with_delivery"] = ((orders["order_status"] == "canceled")
                                    & orders["order_delivered_customer_date"].notna()).astype(int)

print("delivered before purchase :",
      (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print("delivered_missing_date    :", orders["delivered_missing_date"].sum())
print("canceled_with_delivery    :", orders["canceled_with_delivery"].sum())

d["orders"] = orders

delivered before purchase : 0
delivered_missing_date    : 8
canceled_with_delivery    : 6


## Orders: timestamp order checks  (NEW, report only)
A normal order moves purchase → approved → handed to carrier → delivered. This counts rows that break that order, and delivered orders with missing steps. Nothing is changed; the numbers tell you whether any of these dates need caution in your measures.

In [64]:
o = d["orders"]

sequence_checks = {
    "approved before purchase":  o["order_approved_at"] < o["order_purchase_timestamp"],
    "carrier before purchase":   o["order_delivered_carrier_date"] < o["order_purchase_timestamp"],
    "carrier before approved":   o["order_delivered_carrier_date"] < o["order_approved_at"],
    "delivered before carrier":  o["order_delivered_customer_date"] < o["order_delivered_carrier_date"],
}
for label, mask in sequence_checks.items():
    print(f"{label:26s}: {mask.sum():,}")

delivered = o["order_status"] == "delivered"
print("\ndelivered, no approval date :", (delivered & o["order_approved_at"].isna()).sum())
print("delivered, no carrier date  :", (delivered & o["order_delivered_carrier_date"].isna()).sum())

approved before purchase  : 0
carrier before purchase   : 166
carrier before approved   : 1,359
delivered before carrier  : 23

delivered, no approval date : 14
delivered, no carrier date  : 2


## Patch 1: `carrier_date_suspect` (orders)
**Place right after the timestamp-order checks cell** (the one printing `carrier before purchase` etc.).

Flags orders whose carrier handoff date is before the purchase date, or after the customer delivery date. Expected from your earlier output: 166 + 23 = **189**.
Dates are not changed. Use `carrier_date_suspect = 0` in any measure that uses the carrier date (e.g. seller dispatch time).

In [65]:
o = d["orders"]

d["orders"]["carrier_date_suspect"] = (
    (o["order_delivered_carrier_date"] < o["order_purchase_timestamp"])      # handed over before the order existed
    | (o["order_delivered_carrier_date"] > o["order_delivered_customer_date"])  # handed over after it was delivered
).astype(int)

print("carrier_date_suspect:", d["orders"]["carrier_date_suspect"].sum())

carrier_date_suspect: 189


## Orders: coverage and orders without items
Two checks that affect how you build measures later:
- orders with no rows in `order_items` (usually canceled/unavailable), so they carry no revenue
- orders per month, since the first and last months are often incomplete

In [66]:
no_items = ~d["orders"]["order_id"].isin(d["order_items"]["order_id"])
print("orders with no items:", no_items.sum())
display(d["orders"][no_items]["order_status"].value_counts())

monthly = d["orders"].groupby(d["orders"]["order_purchase_timestamp"].dt.to_period("M")).size()
display(monthly)

orders with no items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

## Orders: item and payment flags  (NEW)
- `has_items` = 0 marks orders with no rows in `order_items` (775 in your data, including the single `shipped` one).
- `has_payment` = 0 marks orders with no rows in `payments` (1 in your data).

Use `has_items = 1` as the base for revenue measures that come from `order_items`.

In [67]:
item_ids = set(d["order_items"]["order_id"])
pay_ids  = set(d["payments"]["order_id"])

d["orders"]["has_items"]   = d["orders"]["order_id"].isin(item_ids).astype(int)
d["orders"]["has_payment"] = d["orders"]["order_id"].isin(pay_ids).astype(int)

o = d["orders"]
print("orders without items   :", (o["has_items"] == 0).sum())
print("orders without payments:", (o["has_payment"] == 0).sum())
display(o.loc[o["has_items"] == 0, "order_status"].value_counts())
display(o.loc[o["has_payment"] == 0, ["order_id", "order_status"]])

orders without items   : 775
orders without payments: 1


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

,order_id,order_status
30710,bfbd0f9bdef84302105ad712db648a6c,delivered


## Customers: two keys, one person
`customer_id` is created per order. `customer_unique_id` identifies the person. This shows how many people ordered more than once.

In [68]:
co = d["orders"].merge(d["customers"], on="customer_id", how="left")
per_person = co.groupby("customer_unique_id")["order_id"].nunique()

print("distinct customer_id        :", d["customers"]["customer_id"].nunique())
print("distinct customer_unique_id :", d["customers"]["customer_unique_id"].nunique())
print("share of repeat customers   :", round((per_person > 1).mean() * 100, 2), "%")

distinct customer_id        : 99441
distinct customer_unique_id : 96096
share of repeat customers   : 3.12 %


## Reviews
1. Replace line breaks inside comments with spaces, so the CSV imports cleanly in Power Query.
2. Add `has_comment` **before** anything else touches the text (nulls stay null).
3. Keep one review per order, the most recently answered.
4. Add `review_response_hours` (survey sent to answer submitted).
- `review_creation_date` has no time of day (always midnight), so the gap is only meaningful in **days**. `review_response_hours` is renamed to `review_response_days`.
- After deduplicating on `order_id`, `order_id` is unique but `review_id` is not (one review ID covers several orders). In Power BI, relate reviews to orders on `order_id`.

In [69]:
rev = d["reviews"].copy()

for col in ["review_comment_title", "review_comment_message"]:
    rev[col] = rev[col].str.replace(r"[\r\n]+", " ", regex=True).str.strip()

rev["has_comment"] = rev["review_comment_message"].notna().astype(int)

before = len(rev)
rev = rev.sort_values("review_answer_timestamp").drop_duplicates(subset="order_id", keep="last")
print(f"reviews: {before:,} -> {len(rev):,}")

rev["review_response_days"] = (rev["review_answer_timestamp"]
                               - rev["review_creation_date"]).dt.total_seconds() / 86400

print("answered before survey was sent:", (rev["review_response_days"] < 0).sum())
print("scores outside 1-5             :", (~rev["review_score"].between(1, 5)).sum())
print("order_id unique:", rev["order_id"].is_unique, "| review_id unique:", rev["review_id"].is_unique)

d["reviews"] = rev

reviews: 99,224 -> 98,673
answered before survey was sent: 0
scores outside 1-5             : 0
order_id unique: True | review_id unique: False


## Products
- Fix the `lenght` typos.
- Missing category becomes `"unknown"`.
- Merge the English category name. Two categories are missing from the translation table, so they are mapped by hand.
- Rows with missing weight/size are **kept** (dropping them would orphan rows in `order_items`).
Weight and dimensions are **kept**. New in this cell:
- the English name typo `fashio_female_clothing` is corrected,
- weight `0` (impossible) becomes null, with `weight_was_zero` recording that it happened,
- `product_volume_cm3` = length × height × width, useful for freight and delivery analysis.

In [70]:
prod = d["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
prod["product_category_name"] = prod["product_category_name"].fillna("unknown")

tr = d["category_translation"].assign(
    product_category_name_english=lambda x: x["product_category_name_english"]
        .replace({"fashio_female_clothing": "fashion_female_clothing"}))

n_before = len(prod)
prod = prod.merge(tr, on="product_category_name", how="left")
assert len(prod) == n_before, "merge duplicated rows: translation table has repeated keys"

missing_tr = prod.loc[prod["product_category_name_english"].isna()
                      & (prod["product_category_name"] != "unknown"), "product_category_name"].unique()
print("categories without a translation:", list(missing_tr))

manual = {"pc_gamer": "pc_gamer",
          "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers"}
prod["product_category_name_english"] = (prod["product_category_name_english"]
                                         .fillna(prod["product_category_name"].map(manual))
                                         .fillna(prod["product_category_name"]))

# weight: 0 g is impossible -> null, but remember it happened
prod["weight_was_zero"] = (prod["product_weight_g"] == 0).astype(int)
prod["product_weight_g"] = prod["product_weight_g"].where(prod["product_weight_g"] != 0)

dims = ["product_length_cm", "product_height_cm", "product_width_cm"]
print("dimension == 0 :", {c: int((prod[c] == 0).sum()) for c in dims})
prod["product_volume_cm3"] = prod[dims[0]] * prod[dims[1]] * prod[dims[2]]

print("weight set to null (was 0):", prod["weight_was_zero"].sum())
print("weight missing in total   :", prod["product_weight_g"].isna().sum())

d["products"] = prod

categories without a translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
dimension == 0 : {'product_length_cm': 0, 'product_height_cm': 0, 'product_width_cm': 0}
weight set to null (was 0): 4
weight missing in total   : 6


## Geolocation
About a million rows with many repeats per zip. Steps: drop coordinates outside Brazil, standardise city text, then collapse to one row per zip (mean coordinates, most common city), so joins don't multiply rows.
Differences from the earlier version:
- Out-of-range coordinates are **blanked, not dropped**, so a zip never disappears just because its points were bad.
- Each zip gets the **median** coordinates (one stray point can't drag a mean), the most common city and the most common state.
- The longitude limit is widened so remote islands such as Fernando de Noronha stay valid.
- `clean_text` no longer turns nulls into the text `"nan"`.
- No unnecessary `.copy()` of the 1M-row table: `assign` builds a new frame.


In [71]:
def clean_text(s):
    """lowercase, trim, remove accents; missing values stay missing"""
    s = s.astype("string").str.strip().str.lower()
    return s.map(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode(),
                 na_action="ignore")

ZIP = "geolocation_zip_code_prefix"

g = d["geolocation"]
inside = g["geolocation_lat"].between(-35, 6) & g["geolocation_lng"].between(-75, -28)
print("coordinates outside Brazil (blanked, rows kept):", (~inside).sum())

geo = g.assign(
    geolocation_lat=g["geolocation_lat"].where(inside),
    geolocation_lng=g["geolocation_lng"].where(inside),
    geolocation_city=clean_text(g["geolocation_city"]),
)

def top_value(df, col):
    """most frequent value of `col` within each zip"""
    counts = df.groupby([ZIP, col]).size().reset_index(name="n")
    counts = counts.sort_values([ZIP, "n"], ascending=[True, False])
    return counts.drop_duplicates(ZIP).set_index(ZIP)[col]

out = (geo.groupby(ZIP)[["geolocation_lat", "geolocation_lng"]].median()
          .join(top_value(geo, "geolocation_city"))
          .join(top_value(geo, "geolocation_state")))
out["source_rows"] = geo.groupby(ZIP).size()

d["geolocation"] = out.reset_index()

print("distinct zips in raw file   :", len(raw_geo_zips))
print("zips after cleaning         :", len(d["geolocation"]))
print("zips with no usable coords  :", d["geolocation"]["geolocation_lat"].isna().sum())

coordinates outside Brazil (blanked, rows kept): 29
distinct zips in raw file   : 19015
zips after cleaning         : 19015
zips with no usable coords  : 4


## Customers and sellers: text standardisation
Same `clean_text` helper for city names. State codes are trimmed and upper-cased.

In [72]:
for name, city_col, state_col in [("customers", "customer_city", "customer_state"),
                                  ("sellers", "seller_city", "seller_state")]:
    d[name][city_col]  = clean_text(d[name][city_col])
    d[name][state_col] = d[name][state_col].str.strip().str.upper()

d["customers"]["customer_state"].value_counts().head()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64

##  Orphan zip diagnostic  (NEW)
For customers and sellers, each zip falls into one of three groups:

| Group | Meaning | What happens |
|---|---|---|
| **Missing from the raw geolocation file** | A genuine source gap | Stays **unmatched** (as agreed), `has_geo_match = 0` |
| **Matched, but every point was out of range** | The zip exists but has no usable coordinates. The old version would have dropped these zips and created orphans | Row kept, `has_coordinates = 0` |
| **Matched with coordinates** | Normal | |

The cell also adds `has_geo_match` and `has_coordinates` to `customers` and `sellers`. For maps, filter on `has_coordinates = 1`. For slicing by city or state, use the complete `customer_city` / `customer_state` columns.

In [73]:
geo_zips = set(d["geolocation"][ZIP])
no_coords = set(d["geolocation"].loc[d["geolocation"]["geolocation_lat"].isna(), ZIP])

def zip_report(name, zip_col, city_col, state_col, impact):
    df = d[name]
    missing = ~df[zip_col].isin(geo_zips)
    blank   = df[zip_col].isin(no_coords)

    print(f"===== {name} =====")
    print(f"zip missing from raw geolocation : {missing.sum():>6,} rows | {df.loc[missing, zip_col].nunique():>4,} distinct zips")
    print(f"zip matched, no usable coords    : {blank.sum():>6,} rows | {df.loc[blank, zip_col].nunique():>4,} distinct zips")
    print(impact(df.loc[missing]))
    if missing.any():
        display(df.loc[missing].groupby([zip_col, city_col, state_col]).size()
                  .sort_values(ascending=False).head(10).rename("rows"))

    d[name]["has_geo_match"]   = (~missing).astype(int)
    d[name]["has_coordinates"] = (~missing & ~blank).astype(int)

zip_report("customers", "customer_zip_code_prefix", "customer_city", "customer_state",
           lambda x: f"orders affected              : {d['orders']['customer_id'].isin(x['customer_id']).sum():,}")

zip_report("sellers", "seller_zip_code_prefix", "seller_city", "seller_state",
           lambda x: f"order items affected         : {d['order_items']['seller_id'].isin(x['seller_id']).sum():,}")

===== customers =====
zip missing from raw geolocation :    278 rows |  157 distinct zips
zip matched, no usable coords    :      1 rows |    1 distinct zips
orders affected              : 278


customer_zip_code_prefix  customer_city  customer_state
70686                     brasilia       DF                15
72005                     brasilia       DF                13
71919                     brasilia       DF                10
73255                     brasilia       DF                 7
72300                     brasilia       DF                 6
73369                     brasilia       DF                 5
71884                     brasilia       DF                 5
72002                     brasilia       DF                 5
73401                     brasilia       DF                 5
71676                     brasilia       DF                 4
Name: rows, dtype: int64

===== sellers =====
zip missing from raw geolocation :      7 rows |    7 distinct zips
zip matched, no usable coords    :      0 rows |    0 distinct zips
order items affected         : 253


seller_zip_code_prefix  seller_city      seller_state
02285                   sao paulo        SP              1
07412                   aruja            SP              1
37708                   pocos de caldas  MG              1
71551                   brasilia         DF              1
72580                   brasilia         DF              1
82040                   curitiba         PR              1
91901                   porto alegre     RS              1
Name: rows, dtype: int64

## Seller city check  (NEW, report only)
Seller cities are hand-typed in the source, so some contain digits, slashes, or even a state name instead of a city. This only lists them. Fixing them by hand is your call, and city names from `geolocation` are cleaner if you need them.

In [74]:
state_names = ["minas gerais", "santa catarina", "parana", "bahia", "pernambuco", "ceara",
               "espirito santo", "mato grosso", "mato grosso do sul", "rio grande do sul",
               "rio grande do norte", "maranhao", "piaui", "paraiba", "alagoas", "sergipe",
               "tocantins", "rondonia", "amazonas", "roraima", "amapa"]

sc = d["sellers"]["seller_city"]
odd_chars  = sc.str.contains(r"[0-9/\\@,()]|\s-\s", regex=True, na=False)
state_like = sc.isin(state_names)

print("seller_city with digits/symbols:", odd_chars.sum(), "| looks like a state name:", state_like.sum())
display(d["sellers"].loc[odd_chars | state_like, ["seller_id", "seller_city", "seller_state"]])

seller_city with digits/symbols: 25 | looks like a state name: 5


,seller_id,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP
405,4221a7df464f1fe2955934e30ff3a5a1,bahia,BA
424,c8771b1a10bb99bb34d3c459c5cffb53,tocantins,SP
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ
551,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP
869,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,SP


## Patch 2: `seller_city_clean` and `seller_city_fixed` (sellers)
**Place right after the seller-city check cell** (needs the cleaned `geolocation` table and `clean_text` already applied to `seller_city`).

How it works:
1. Cut the text at `/`, `\`, `,`, ` - ` or `(`, keeping the first part (`lages - sc` → `lages`). A hyphen without spaces is left alone (`embu-guacu`).
2. One manual abbreviation: `sbc` → `sao bernardo do campo`.
3. If what is left is unusable (contains digits or `@`, is a state name, or is 2 letters or fewer), take the city from `geolocation` using the seller's zip. If the zip has no match, use `unknown`.
4. `seller_city_fixed` = 1 wherever the value changed. The original `seller_city` is untouched.

Review the printed before/after table. The state-name rule can be wrong for a real town that shares a state's name (e.g. a town called Tocantins in MG), so check the `via_geolocation` rows.

In [75]:
STATE_NAMES = ["minas gerais", "santa catarina", "parana", "bahia", "pernambuco", "ceara",
               "espirito santo", "mato grosso", "mato grosso do sul", "rio grande do sul",
               "rio grande do norte", "maranhao", "piaui", "paraiba", "alagoas", "sergipe",
               "tocantins", "rondonia", "amazonas", "roraima", "amapa"]
ABBREVIATIONS = {"sbc": "sao bernardo do campo"}

original = d["sellers"]["seller_city"].astype("string")

# 1-2. keep the text before the first separator, then expand known abbreviations
cut = (original.str.split(r"\s*(?:/|\\|,|\s-\s|\(|[\r\n])", n=1, regex=True).str[0]
               .str.strip()
               .replace(ABBREVIATIONS))

# 3. values that cannot be a city name -> use the city from geolocation by zip
unusable = (cut.isna() | cut.str.contains(r"[0-9@]", regex=True, na=False)
            | cut.isin(STATE_NAMES) | (cut.str.len() <= 2))

geo_city = d["geolocation"].set_index("geolocation_zip_code_prefix")["geolocation_city"]
fallback = d["sellers"]["seller_zip_code_prefix"].map(geo_city)

clean = cut.where(~unusable, fallback).fillna("unknown")

# 4. flag what changed and add the new columns
changed = (clean != original).fillna(False)
d["sellers"]["seller_city_clean"] = clean
d["sellers"]["seller_city_fixed"] = changed.astype(int)

review = pd.DataFrame({"before": original[changed], "after": clean[changed],
                       "seller_state": d["sellers"].loc[changed, "seller_state"],
                       "via_geolocation": unusable[changed]})
display(review)
print("sellers changed:", changed.sum(), "| distinct cities before:", original.nunique(),
      "| after:", clean.nunique(), "| set to 'unknown':", (clean == "unknown").sum())

,before,after,seller_state,via_geolocation
78,lages - sc,lages,SC,False
237,auriflama/sp,auriflama,SP,False
246,sao paulo / sao paulo,sao paulo,SP,False
405,bahia,paulo afonso,BA,True
517,04482255,rio de janeiro,RJ,True
551,"novo hamburgo, rio grande do sul, brasil",novo hamburgo,RS,False
622,cariacica / es,cariacica,ES,False
826,sao paulo - sp,sao paulo,SP,False
869,sbc/sp,sao bernardo do campo,SP,False
874,arraial d'ajuda (porto seguro),arraial d'ajuda,BA,False


sellers changed: 34 | distinct cities before: 609 | after: 583 | set to 'unknown': 0


## Payments and order items: sanity checks
These print counts instead of stopping the notebook, so you can decide what to do with anything unusual.
- `is_zero_payment` = 1 when `payment_value` is 0 (your 9 rows).
- `installments_missing` = 1 for credit-card payments with 0 installments (your 2 rows). Values are left untouched.
- `not_defined` payment type is relabelled `unknown`.
- The flagged rows are shown with their order status so you can see what kind of orders they belong to.

In [76]:
pay = d["payments"]
display(pay["payment_type"].value_counts())
print("payment_value == 0:", (pay["payment_value"] == 0).sum())
print("installments == 0 :", (pay["payment_installments"] == 0).sum())

d["payments"]["is_zero_payment"]      = (pay["payment_value"] == 0).astype(int)
d["payments"]["installments_missing"] = ((pay["payment_type"] == "credit_card")
                                         & (pay["payment_installments"] == 0)).astype(int)
d["payments"]["payment_type"] = pay["payment_type"].replace({"not_defined": "unknown"})

flagged = d["payments"][(d["payments"]["is_zero_payment"] == 1) | (d["payments"]["installments_missing"] == 1)]
display(flagged.merge(d["orders"][["order_id", "order_status"]], on="order_id", how="left"))

items = d["order_items"]
print("\nitems: price <= 0     :", (items["price"] <= 0).sum())
print("items: freight < 0    :", (items["freight_value"] < 0).sum())
print("items: freight == 0   :", (items["freight_value"] == 0).sum())
print("items: freight > price:", (items["freight_value"] > items["price"]).sum())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

payment_value == 0: 9
installments == 0 : 2


,order_id,payment_sequential,payment_type,payment_installments,payment_value,is_zero_payment,installments_missing,order_status
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00,1,0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00,1,0,shipped
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00,1,0,delivered
3,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,0,1,delivered
4,4637ca194b6387e2d538dc89b124b0ee,1,unknown,1,0.00,1,0,canceled
5,00b1cb0320190ca0daa2c88b35206009,1,unknown,1,0.00,1,0,canceled
6,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00,1,0,delivered
7,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00,1,0,shipped
8,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,0,1,delivered
9,c8c528189310eaa44a745b8d9d26908b,1,unknown,1,0.00,1,0,canceled



items: price <= 0     : 0
items: freight < 0    : 0
items: freight == 0   : 383
items: freight > price: 4124


## Referential integrity (orphan keys)
Every child key should exist in its parent table. Non-zero counts need investigating before you build the Power BI model.

In [77]:
def orphans(child, col, parent, pcol=None):
    pcol = pcol or col
    n = (~d[child][col].isin(d[parent][pcol])).sum()
    print(f"{child}.{col:28s} not in {parent}: {n:,}")

orphans("orders", "customer_id", "customers")
orphans("order_items", "order_id", "orders")
orphans("order_items", "product_id", "products")
orphans("order_items", "seller_id", "sellers")
orphans("payments", "order_id", "orders")
orphans("reviews", "order_id", "orders")
orphans("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix")
orphans("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix")

orders.customer_id                  not in customers: 0
order_items.order_id                     not in orders: 0
order_items.product_id                   not in products: 0
order_items.seller_id                    not in sellers: 0
payments.order_id                     not in orders: 0
reviews.order_id                     not in orders: 0
customers.customer_zip_code_prefix     not in geolocation: 278
sellers.seller_zip_code_prefix       not in geolocation: 7


## Final null check and export
Expected null columns now: `orders` (the date columns and the derived delivery columns), `reviews` (comment text), `products` (the 610 products with no category info, plus weight and size), `geolocation` (zips without usable coordinates). Everything else should be empty.

The translation table is already merged into `products`, so it is not exported. In Power Query set zip codes and IDs to **Text** and the file origin to **UTF-8 (65001)**.

In [78]:
for name, df in d.items():
    n = df.isnull().sum()
    print(f"{name:22s}", n[n > 0].to_dict())

for name, df in d.items():
    if name == "category_translation":
        continue
    df.to_csv(CLEAN / f"{name}_clean.csv", index=False, encoding="utf-8")
    print(f"saved {name}_clean.csv  ({len(df):,} rows, {df.shape[1]} columns)")

customers              {}
geolocation            {'geolocation_lat': 4, 'geolocation_lng': 4}
orders                 {'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965, 'delivery_days': 2965, 'delivery_delay_days': 2965, 'is_late': 2965}
order_items            {}
payments               {}
reviews                {'review_comment_title': 87123, 'review_comment_message': 57898}
products               {'product_name_length': 610, 'product_description_length': 610, 'product_photos_qty': 610, 'product_weight_g': 6, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2, 'product_volume_cm3': 2}
sellers                {}
category_translation   {}
saved customers_clean.csv  (99,441 rows, 7 columns)
saved geolocation_clean.csv  (19,015 rows, 6 columns)
saved orders_clean.csv  (99,441 rows, 17 columns)
saved order_items_clean.csv  (112,650 rows, 7 columns)
saved payments_clean.csv  (103,886 rows, 7 columns)
saved reviews_clean.cs